# Process Extracted 3-D RI into Mesh Labels
Loads the 3-D masked RI volume produced by `cell_ri_extract_3d.ipynb` and:
1. Determines image parameters (pixel sizes, gel cut-off Z-plane).
2. Pads the volume: voxels above the gel line → **air** RI, below → **gel** RI.
3. Saves the labelled volume as a `.mat` file ready for meshing.

In [ ]:
# Imports
import os
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt

## I/O Allocation

In [ ]:
# Path variables
repo_path      = "/path/to/repository/"   # <-- replace
export_address = "data/mesh_label/"
access_address = "data/extracted_ri/"

def output_address(subfolder="data/", repository=repo_path):
    return f"{repository}{subfolder}"

def modified_output_address(file_name):
    name_part = file_name.split(".")[0]
    return f"{name_part}_label_3d.mat"

In [ ]:
# Accessing the file
file_name   = "Test_extracted_3d.mat"   # <-- replace
access_path = f"{repo_path}{access_address}{file_name}"
print(f"Accessing: {access_path}")

content = sp.io.loadmat(access_path)
print("Keys:", content.keys())

In [ ]:
# Retrieve 3-D volume  — shape (Z, Y, X)
extract = content['masked_ri']

print(f"Volume shape (Z, Y, X): {extract.shape}")
print(f"dtype: {extract.dtype}  |  max RI: {extract.max():.6f}")

# Sanity: verify exactly 3 dimensions
if extract.ndim != 3:
    raise ValueError(
        f"Expected a 3-D array, got {extract.ndim}D. "
        "Re-run cell_ri_extract_3d.ipynb to produce the correct output.")

# Show middle slice
mid_z = extract.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(extract[mid_z], cmap="viridis")
plt.colorbar(label="RI")
plt.title(f"Middle Z-slice (z={mid_z})")
plt.tight_layout()
plt.show()

## Image Parameters
Enter physical dimensions of the acquisition volume.

In [ ]:
# Physical parameters — adjust to match your microscope settings
length_um = 35.7014   # physical width  of FOV in µm  (X direction)
width_um  = 35.7014   # physical height of FOV in µm  (Y direction)
depth_um  = 212.0     # physical depth  of FOV in µm  (Z direction)
unit      = 'um'

n_z, n_y, n_x = extract.shape   # voxel counts

pix_size_x = length_um / n_x   # µm per voxel in X
pix_size_y = width_um  / n_y   # µm per voxel in Y
pix_size_z = depth_um  / n_z   # µm per voxel in Z

print(f"Volume voxel grid   : {n_z} (Z) × {n_y} (Y) × {n_x} (X)")
print(f"Voxel size X        : {pix_size_x:.5f} {unit}")
print(f"Voxel size Y        : {pix_size_y:.5f} {unit}")
print(f"Voxel size Z        : {pix_size_z:.5f} {unit}")

## Gel-Line Detection
Find the lowest Z-index at which the cell is still present.  
All slices at or above that index belong to the **air** region;  
slices below belong to the **gel** region.

In [ ]:
def find_cutoff_z(vol):
    """
    Scan Z-slices from bottom (high Z) upward.
    Return the index of the deepest Z-slice that still contains
    non-zero RI values (i.e., cell material).
    
    vol: ndarray, shape (Z, Y, X)
    """
    for z in range(vol.shape[0] - 1, -1, -1):
        if np.sum(vol[z]) > 0:
            return z
    return 0  # fallback: entire stack is empty

cutoff_z = find_cutoff_z(extract)
print(f"Gel-line Z-index (cutoff): {cutoff_z}  "
      f"({cutoff_z * pix_size_z:.3f} {unit} from top of stack)")

## 3-D Padding
Fill the empty voxels with the appropriate background RI:
- **Air** (above gel line): RI = 1.30
- **Gel** (below gel line): RI = 1.34

In [ ]:
air_ri = 1.30
gel_ri = 1.34

def pad_3d(vol, air, gel, cutoff_z):
    """
    Fill zero-RI voxels with background medium values.

    Parameters
    ----------
    vol      : ndarray (Z, Y, X), modified in-place
    air      : RI value for medium above gel line
    gel      : RI value for medium below (and at) gel line
    cutoff_z : Z-index of the gel surface

    Returns
    -------
    vol : the modified volume
    """
    vol = vol.copy()   # avoid mutating the source array

    # Slices 0 … cutoff_z  →  air region
    region_air = vol[:cutoff_z + 1]
    region_air[region_air == 0] = air
    vol[:cutoff_z + 1] = region_air

    # Slices cutoff_z+1 … end  →  gel region
    region_gel = vol[cutoff_z + 1:]
    region_gel[region_gel == 0] = gel
    vol[cutoff_z + 1:] = region_gel

    return vol

out_label = pad_3d(extract, air_ri, gel_ri, cutoff_z)

print(f"Padded volume shape : {out_label.shape}")
print(f"RI range            : [{out_label.min():.4f}, {out_label.max():.4f}]")

# Visualise padded middle slice
mid_z = out_label.shape[0] // 2
plt.figure(figsize=(7, 5))
plt.imshow(out_label[mid_z], cmap="viridis")
plt.colorbar(label="RI (padded)")
plt.title(f"Middle Z-slice (z={mid_z}) — padded mesh labels")
plt.tight_layout()
plt.show()

## Saving Output File

In [ ]:
output_path = output_address(export_address, repo_path)
output_name = modified_output_address(file_name)

try:
    sp.io.savemat(
        f"{output_path}{output_name}",
        {"masked_ri": out_label}
    )
    print(f"Mesh-label volume saved → {output_path}{output_name}")
except Exception as e:
    print(f"Error saving: {e}")
    raise